# Фундаментальная модель: Chronos

**Chronos** (Amazon) — модель, предобученная на огромном количестве временных рядов. Используется
**zero-shot**: на наших данных не дообучается, получает на вход только историю ряда территории
и сразу выдаёт прогноз. Это то, чего просят в задаче конкурса (сравнить классику, ML и фундаментальную модель).

Схема проверки та же, что в остальных ноутбуках.

In [6]:
%load_ext autoreload
%autoreload 2
from forecasting import *

df_general = pd.read_parquet('../data/df_general.parquet')
dates_all = pd.to_datetime(np.sort(df_general['date'].unique()))
EVAL_TARGETS = list(dates_all[-6:])

rng = np.random.default_rng(42)
sample_ids = rng.choice(df_general['territory_id'].unique(), 300, replace=False)

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется: {device}')

Используется: mps


## Как используется Chronos

- Модель `chronos-t5-small` (самая маленькая версия, быстро считается на ноутбуке).
- На вход идёт **только ряд расходов самой территории** (12–23 точки), без других признаков.
- Модель строит 20 случайных траекторий, в качестве прогноза берётся их **медиана**.
- Перед каждым прогнозом фиксируется seed, чтобы цифры воспроизводились.
- Прогнозируется сразу уровень (рубли), перевод в относительный прирост не нужен: Chronos сам делит ряд
  на его среднее перед работой.

In [10]:
pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=f'{device}',
    torch_dtype=torch.float32
)
def chronos_forecast(train, horizon, batch_size=50):
    territories = train['territory_id'].unique()
    res = []
    
    for i in range(0, len(territories), batch_size):
        batch_ids = territories[i:i+batch_size]
        contexts = []
        
        for tid in batch_ids:
            ts = (train[train['territory_id'] == tid]
                  .sort_values('date')['total_spending']
                  .values
                  .astype(float))
            contexts.append(torch.tensor(ts, dtype=torch.float32))
        
        forecast = pipeline.predict(
            contexts,
            prediction_length=horizon,
            num_samples=20
        )
        
        median_forecast = np.median(forecast.numpy(), axis=1)
        
        for j, tid in enumerate(batch_ids):
            res.append({
                'territory_id': int(tid),
                'total_spending': float(median_forecast[j, horizon - 1])
            })
    
    return pd.DataFrame(res)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [11]:
def chronos_wrapped(train, horizon, target_date, **kw):
    torch.manual_seed(42)
    return chronos_forecast(train, horizon)

print('Все 2190 территорий:')
res_chronos = run_all(df_general, chronos_wrapped, EVAL_TARGETS)
display(res_chronos)

print('300 территорий (как для Prophet):')
res_chronos_300 = run_all(df_general, chronos_wrapped, EVAL_TARGETS, eval_ids=sample_ids)
display(res_chronos_300)

Все 2190 территорий:


,MAE,WAPE,R2
horizon,,,
1,2190.362,0.066,0.936
3,2898.255,0.086,0.883
6,4991.409,0.153,0.733
12,6176.881,0.191,0.646


300 территорий (как для Prophet):


,MAE,WAPE,R2
horizon,,,
1,2116.388,0.066,0.937
3,2743.909,0.085,0.888
6,4667.492,0.149,0.747
12,5783.314,0.187,0.660


## Результат

| MAE, все 2190 территорий | 1 мес | 3 мес | 6 мес | 12 мес |
|---|---|---|---|---|
| Chronos-small | 2190 | 2898 | 4991 | 6177 |
| WAPE | 6,6% | 8,6% | 15,3% | 19,1% |
| R² | 0,94 | 0,88 | 0,73 | 0,65 |

На 1 месяц Chronos примерно на уровне naive (2023), а с ростом горизонта сильно проигрывает
seasonal naive и итоговой модели (см. `02_forecast.ipynb`).

**Почему так:** у него всего 12–23 точки одного ряда, и модель не может понять, что есть годовая
сезонность (за 12 месяцев её видно один раз). Дополнительных признаков (календарь, макро) она не получает.

## Выводы и ограничения
- Фундаментальная модель без дообучения проигрывает простой сезонной модели на таких коротких рядах.
- Проверен только Chronos-small; более крупные версии не пробовала.
- TimesFM пробовала запустить, но он слишком долго считался на ноутбуке, поэтому в сравнение не вошёл.